In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import warnings
import matplotlib.pyplot as plt
import time
warnings.filterwarnings("ignore")

# 1. LOAD DATA DAN SCALER
base_path = './split/'

train = pd.read_csv(base_path + '90training_normalized.csv')
test  = pd.read_csv(base_path + '10testing_normalized.csv')
test_asli = pd.read_csv(base_path + '9testing.csv') 

print("=== 5 Baris Pertama Data Training (train) ===")
print(train.head())
print("\n=== 5 Baris Pertama Data Testing (test) ===")
print(test.head())
print("\n=== 5 Baris Pertama Data Asli (test_asli) ===")
print(test_asli.head())
print("=============================================\n")

X_train = train.drop(columns=['Produksi'])
y_train = train['Produksi']

X_test = test.drop(columns=['Produksi'])
y_test = test['Produksi']

scaler_y = joblib.load(base_path + '90training_scaler_y.save')

# Preprocessing data test_asli untuk output CSV
bulan_map = {
    'Januari':1, 'Februari':2, 'Maret':3, 'April':4,
    'Mei':5, 'Juni':6, 'Juli':7, 'Agustus':8,
    'September':9, 'Oktober':10, 'November':11, 'Desember':12
}
test_asli[['Nama_Bulan', 'Tahun']] = test_asli['Periode'].str.split(' ', expand=True)
test_asli['Bulan'] = test_asli['Nama_Bulan'].map(bulan_map)
test_asli['Tahun'] = test_asli['Tahun'].astype(int)
test_asli.drop(columns=['Periode', 'Nama_Bulan'], inplace=True)


# 2. FUNGSI EVALUASI (FITNESS FUNCTION PSO MENGGUNAKAN RMSE)
def evaluate_svr(params):
    C, epsilon, gamma = params
    try:
        model = SVR(kernel='rbf', C=C, epsilon=epsilon, gamma=gamma)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Denormalisasi agar PSO mengevaluasi error dalam satuan asli (Ton)
        y_pred_asli = scaler_y.inverse_transform(y_pred.reshape(-1,1)).flatten()
        y_test_asli = scaler_y.inverse_transform(y_test.values.reshape(-1,1)).flatten()
        
        # Mencegah nilai prediksi negatif (jika ada)
        y_pred_asli = np.clip(y_pred_asli, 0, None)
        
        # Hitung RMSE
        rmse_score = np.sqrt(mean_squared_error(y_test_asli, y_pred_asli))
        
        return rmse_score, y_pred 
    except:
        return float('inf'), None


# 3. FUNGSI EXPORT HASIL CSV
def export_hasil_terbaik(n_particles, global_best, global_best_pred):
    # Denormalisasi Prediksi Testing
    y_pred_asli = scaler_y.inverse_transform(global_best_pred.reshape(-1,1)).flatten()
    y_pred_asli = np.clip(y_pred_asli, 0, None)
    y_test_asli = scaler_y.inverse_transform(y_test.values.reshape(-1,1)).flatten()

    # Hitung RMSE untuk ditampilkan saat proses export
    current_rmse = np.sqrt(mean_squared_error(y_test_asli, y_pred_asli))

    # Debug: Print beberapa nilai y_test_asli dan y_pred_asli
    print("Contoh y_test_asli:", y_test_asli[:10])
    print("Contoh y_pred_asli:", y_pred_asli[:10])
    print("RMSE (denormalized): {:.4f}".format(current_rmse))

    # Gabung dan Simpan ke CSV
    hasil = test_asli.copy()
    hasil['Produksi_Asli'] = y_test_asli.round(2)
    hasil['Prediksi_Ton']  = y_pred_asli.round(2)
    hasil = hasil[['Tahun', 'Bulan', 'Kabupaten/Kota', 'Produksi_Asli', 'Prediksi_Ton']]
    
    nama_file_csv = f'hasil_terbaik_rmse_{n_particles}_partikel.csv'
    hasil.to_csv(nama_file_csv, index=False)


# 4. ALGORITMA PSO UTAMA DENGAN RESUME
def pso_auto_resume(n_particles, target_iter, timeout=10):
    np.random.seed(42)
    # Nama checkpoint diubah agar tidak bentrok dengan file lama yang pakai MAPE
    nama_checkpoint = f'pso_state_rmse_{n_particles}_partikel.save'
    
    lb = np.array([1, 0.000001, 0.00001])
    ub = np.array([1000, 0.1, 100])
    w, c1, c2 = 0.7, 1.5, 1.5

    # Cek apakah ada file checkpoint sebelumnya
    if os.path.exists(nama_checkpoint):
        print(f">>> File checkpoint ditemukan! Memuat data {n_particles} partikel...")
        state = joblib.load(nama_checkpoint)
        
        start_iter = state['iterasi_terakhir']
        particles = state['particles']
        velocities = state['velocities']
        personal_best = state['personal_best']
        personal_best_score = state['personal_best_score']
        global_best = state['global_best']
        global_best_score = state['global_best_score']
        global_best_pred = state['global_best_pred']
        rmse_history = state['rmse_history']
        
        print(f">>> Melanjutkan dari iterasi ke-{start_iter + 1} menuju {target_iter}...\n")
    else:
        print(f">>> Memulai PSO dari awal untuk {n_particles} partikel (Target: Minimalisasi RMSE)...\n")
        start_iter = 0
        particles = np.random.uniform(lb, ub, (n_particles, 3))
        velocities = np.zeros((n_particles, 3))
        personal_best = particles.copy()
        personal_best_score = np.array([float('inf')] * n_particles)
        global_best = None
        global_best_score = float('inf')
        global_best_pred = None
        rmse_history = []

    if start_iter >= target_iter:
        print("Target iterasi sudah tercapai di run sebelumnya. Tidak ada iterasi baru dijalankan.")
        return rmse_history

    # Looping Iterasi
    for i in range(start_iter, target_iter):
        print(f"iterasi ke {i+1}/{target_iter} :")
        
        for j in range(n_particles):
            start = time.time()
            score, pred = evaluate_svr(particles[j])
            elapsed = time.time() - start
            
            if elapsed > timeout:
                score, pred = float('inf'), None

            if score < personal_best_score[j]:
                personal_best[j] = particles[j]
                personal_best_score[j] = score

                if score < global_best_score:
                    global_best = particles[j]
                    global_best_score = score
                    global_best_pred = pred
            
            score_tampil = f"{score:.4f}" if score != float('inf') else "inf"
            print(f"partikel {j+1}/{n_particles}, RMSE : {score_tampil}, waktu: {elapsed:.4f} s.")

        rmse_history.append(global_best_score)

        # Update Posisi & Velocity
        for j in range(n_particles):
            r1, r2 = np.random.rand(), np.random.rand()
            velocities[j] = (
                w * velocities[j]
                + c1 * r1 * (personal_best[j] - particles[j])
                + c2 * r2 * (global_best - particles[j])
            )
            particles[j] += velocities[j]
            particles[j] = np.clip(particles[j], lb, ub)

        print(f"--- Iterasi {i+1} Selesai | Global Best RMSE: {global_best_score:.4f} ---\n")

        # SIMPAN CHECKPOINT SETIAP ITERASI 
        state = {
            'iterasi_terakhir': i + 1,
            'particles': particles,
            'velocities': velocities,
            'personal_best': personal_best,
            'personal_best_score': personal_best_score,
            'global_best': global_best,
            'global_best_score': global_best_score,
            'global_best_pred': global_best_pred,
            'rmse_history': rmse_history
        }
        joblib.dump(state, nama_checkpoint)
        
        # Ekspor CSV terbaru
        export_hasil_terbaik(n_particles, global_best, global_best_pred)

    return rmse_history


# 5. RUN PROGRAM UTAMA
JUMLAH_PARTIKEL = 100
TARGET_ITERASI = 150

history = pso_auto_resume(n_particles=JUMLAH_PARTIKEL, target_iter=TARGET_ITERASI)

# 6. SIMPAN MODEL TERBAIK DAN EVALUASI FINAL
# Ambil parameter terbaik hasil PSO terakhir
state = joblib.load(f'pso_state_rmse_{JUMLAH_PARTIKEL}_partikel.save')
best_params = state['global_best']
C_best, epsilon_best, gamma_best = best_params

print('\nHASIL AKHIR PARAMETER (Optimasi RMSE):')
print('C = {:.4f}'.format(C_best))
print('e = {:.6f}'.format(epsilon_best))
print('gamma = {:.4f}'.format(gamma_best))

# Train ulang model terbaik di seluruh data training
model_best = SVR(kernel='rbf', C=C_best, epsilon=epsilon_best, gamma=gamma_best)
model_best.fit(X_train, y_train)

# Nama model disesuaikan
joblib.dump(model_best, 'model_svr_rbf_best_rmse.save')
print('Model terbaik berhasil disimpan: model_svr_rbf_best_rmse.save')

# Prediksi training dan testing
y_train_pred = model_best.predict(X_train)
y_test_pred = model_best.predict(X_test)

# Denormalisasi hasil prediksi dan target
y_train_pred_asli = scaler_y.inverse_transform(y_train_pred.reshape(-1,1)).flatten()
y_train_asli = scaler_y.inverse_transform(y_train.values.reshape(-1,1)).flatten()
y_test_pred_asli = scaler_y.inverse_transform(y_test_pred.reshape(-1,1)).flatten()
y_test_asli = scaler_y.inverse_transform(y_test.values.reshape(-1,1)).flatten()

# Mencegah nilai negatif
y_train_pred_asli = np.clip(y_train_pred_asli, 0, None)
y_test_pred_asli = np.clip(y_test_pred_asli, 0, None)

# --- EVALUASI FINAL SESUAI REVISI (HANYA RMSE DAN R2) ---

print('\nEvaluasi Training :')
print('RMSE: {:.4f}'.format(np.sqrt(mean_squared_error(y_train_asli, y_train_pred_asli))))
print('R2:   {:.4f}'.format(r2_score(y_train_asli, y_train_pred_asli)))

print('\nEvaluasi Testing :')
print('RMSE: {:.4f}'.format(np.sqrt(mean_squared_error(y_test_asli, y_test_pred_asli))))
print('R2:   {:.4f}'.format(r2_score(y_test_asli, y_test_pred_asli)))

# Tampilkan Grafik Konvergensi Akhir (Sumbu Y sekarang RMSE)
if len(history) > 0:
    plt.figure()
    plt.plot(history)
    plt.title(f"Konvergensi PSO ({JUMLAH_PARTIKEL} Partikel) - Target RMSE")
    plt.xlabel("Iterasi")
    plt.ylabel("RMSE (Ton)")
    plt.grid()
    plt.show()

print("\n>>> PROSES SELESAI <<<")
print(f"Hasil prediksi tersimpan di: hasil_terbaik_rmse_{JUMLAH_PARTIKEL}_partikel.csv")